In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number
from helpers import get_table, get_bronze
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from functools import reduce
print(pyspark.__version__)

3.5.8


In [2]:
builder = (
    SparkSession.builder
    .appName("user_bronze")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.postgresql:postgresql:42.7.3"]
).getOrCreate()

26/04/27 18:35:14 WARN Utils: Your hostname, CVPThuyTTD11-1 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/27 18:35:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/thuyttd11/.local/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thuyttd11/.ivy2/cache
The jars for the packages stored in: /home/thuyttd11/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e9ff0360-a811-4ccc-96bd-41762727644f;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 329ms :: artifacts dl 22ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.3 from central in [default]
	------------------------

# Silver subscriptions
+ Read Bronze subscriptions data
+ Cast and align data types: subscription_id, user_id, plan_id, current_mrr, with user_id aligned to silver_users.user_id and plan_id aligned to silver_subscription_plans.plan_id
+ Convert and normalize datetime fields: start_date, end_date, created_at, with consistent timezone handling for timestamp columns
+ Clean text fields: trim whitespace and convert empty strings to null for all string columns
+ Standardize categorical values: normalize status to lowercase and restrict to valid values active, cancelled, expired, trial
+ Validate data quality and business rules: required fields subscription_id, user_id, plan_id, start_date, status; referential integrity for user_id and plan_id; valid date logic such as start_date <= end_date when end_date exists; current_mrr = 0 for trial and current_mrr > 0 for paid subscriptions; duplicates removed and uniqueness enforced based on the correct business grain, not blindly one row per subscription_id unless this is a current-state table
+ Derive business columns: start_year, start_month, start_date_only, tenure_days, tenure_months, is_active_flag, is_trial_flag, is_paid_flag, is_cancelled_flag, subscription_start_month, subscription_end_month, normalized_monthly_mrr, and optional previous_status if source history exists
+ Quarantine invalid records: rows failing key, foreign key, status, date, MRR, or duplicate/grain validation
+ Persist Silver output: write the cleaned dataset to Delta as silver_subscriptions
+ Important: Final quarantine dataframe

## Read Bronze subscriptions data

In [3]:
# Bronze layer
subscriptions_bronze = "/home/thuyttd11/subscription-analytics/spark-data/bronze/subscriptions"
df_bronze_subscriptions = get_bronze(subscriptions_bronze, spark=spark).drop("source_identifier", "batch_id", "id")
df_bronze_subscriptions.show(5)

26/04/27 18:35:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+
|subscription_id|user_id|plan_id|start_date|  end_date|   status|current_mrr|         created_at|         ingest_time|
+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+
|              1|  15555|      4|2026-04-12|      NULL|   active|      41.87|2026-04-12 11:03:43|2026-04-22 11:29:...|
|              2|  25859|      2|2024-01-06|      NULL|   active|      10.40|2024-01-06 20:34:33|2026-04-22 11:29:...|
|              3|    689|      3|2025-09-09|2025-11-02|cancelled|     370.16|2025-09-09 21:12:17|2026-04-22 11:29:...|
|              4|   9263|     11|2024-07-06|      NULL|   active|      23.62|2024-07-06 15:16:42|2026-04-22 11:29:...|
|              5|  38928|     10|2023-12-01|      NULL|   active|       8.31|2023-12-01 12:07:00|2026-04-22 11:29:...|
+---------------+-------+-------+----------+----

## Remove duplicates based on the correct business grain
If this is a current-state subscriptions table, use subscription_id as the uniqueness key and keep the latest row.
Indeed, we can even use the upsert for this purpose, but, this is still a greate idea.


In [4]:
# df_bronze_subscriptions1 = df_bronze_subscriptions.dropDuplicates(["subscription_id"])
# df_bronze_subscriptions1.count()

# Window: latest row per subscription_id
w = Window.partitionBy("subscription_id", "created_at").orderBy(
    col("ingest_time").desc()
)

# Keep latest row ingested
df1 = (
    df_bronze_subscriptions
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Duplicate rows to quarantine
df1_quarantine = (
    df_bronze_subscriptions
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") > 1)
    .drop("rn")
)


In [5]:
# df1.count(), df1_quarantine.count()

## Quarantine NULL (except end_date)

In [6]:
df2 = df1.filter(
    col("subscription_id").isNotNull() &
    col("user_id").isNotNull() &
    col("plan_id").isNotNull() &
    col("start_date").isNotNull() &
    col("status").isNotNull()
)
df2_quarantine = df1.subtract(df2)

In [7]:
# df2.count(), df2_quarantine.count()

## Cast and align data types

In [8]:
df3 = df2.select(
col("subscription_id").cast("bigint").alias("subscription_id"),
        col("user_id").cast("bigint").alias("user_id"),
        col("plan_id").cast("bigint").alias("plan_id"),
        col("start_date").cast("date").alias("start_date"),
        col("end_date").cast("date").alias("end_date"),
        col("status").cast("string").alias("status"),
        col("current_mrr").cast("decimal(10,2)").alias("current_mrr"),
        col("created_at").cast("timestamp").alias("created_at"),
        col("ingest_time").cast("timestamp").alias("ingest_time"),
)
df3.show(2)

[Stage 12:===========================================>              (3 + 1) / 4]

+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+
|subscription_id|user_id|plan_id|start_date|  end_date|   status|current_mrr|         created_at|         ingest_time|
+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+
|              1|  15555|      4|2026-04-12|      NULL|   active|      41.87|2026-04-12 11:03:43|2026-04-22 11:29:...|
|              3|    689|      3|2025-09-09|2025-11-02|cancelled|     370.16|2025-09-09 21:12:17|2026-04-22 11:29:...|
+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+
only showing top 2 rows



## Standardize categorical values

In [9]:
valid_status = ["active", "cancelled", "expired", "trial"]

df4_temp = (
    df3
    .withColumn("status", F.lower(F.trim(col("status"))))
)

df4 = df4_temp.filter(
    col("status").isin(valid_status)
)

df4_quarantine = df4_temp.filter(
    col("status").isNull() | (~col("status").isin(valid_status))
)
df4.select("status").distinct().show()
df4_quarantine.select("status").distinct().show()

+---------+
|   status|
+---------+
|    trial|
|cancelled|
|   active|
|  expired|
+---------+



[Stage 25:===========================================>              (3 + 1) / 4]

+------+
|status|
+------+
+------+



## Validate data quality and business rules

### Reference keys

Only validate with users_silver, not plans_silver yet.

In [10]:
df_user_silver = (
    spark.read
    .format("delta")
    .load("./silver/users")
)

df_valid_users = (
    df_user_silver
    .select("user_id")
    .distinct()
)

# Valid rows: user_id exists in silver users
df5 = (
    df4
    .join(df_valid_users, on="user_id", how="left_semi")
)

# Quarantine rows: user_id does NOT exist in silver users
df5_quarantine = (
    df4
    .join(df_valid_users, on="user_id", how="left_anti")
)

df5.count(), df5_quarantine.count()

(250000, 0)

### Validate date logic

In [11]:
df6 = df5.filter(
    col("end_date").isNull() | (col("start_date") <= col("end_date"))
)
# Quarantine rows: invalid date logic
df6_quarantine = df5.filter(
    col("end_date").isNotNull() & (col("start_date") > col("end_date"))
)

df6.count(), df6_quarantine.count()

(250000, 0)

### Validate current_mrr business rule

current_mrr: is the amount of recurring revenue that one subscription is currently generating per month.

if status = 'trial' → current_mrr = 0

otherwise (active, cancelled, expired) → current_mrr > 0

Indeed, the current data generated already consider the users and plans dataframes, so, I dont think there are too many wrong values here, just check because it might be wrong in real life scenarios.

In [12]:
# Clean rows: valid MRR rule
df7 = df6.filter(
    ((col("status") == "trial") & (col("current_mrr") == 0)) |
    ((col("status").isin("active", "cancelled", "expired")) & (col("current_mrr") > 0))
)
# Quarantine rows: invalid MRR rule
df7_quarantine = df6.filter(
    ((col("status") == "trial") & (col("current_mrr") != 0)) |
    ((col("status").isin("active", "cancelled", "expired")) & (col("current_mrr") <= 0))
)
df7.count(),df7_quarantine.count()

(226095, 23905)

#### Comment for current_mrr business rule

There are status which is not trial, but still get current_mrr=0.0, but why?
Although I look into the logic in generating subscriptions data, especially in the 
```python
# MRR logic
if plan["tier"] == "Free Trial" or status == "trial":
    mrr = 0
else:
    mrr = monthly_mrr(plan["price"], plan["billing_cycle"]),

The reason maybe in the else
```

In [13]:
# values = (
#     df7
#     .select("current_mrr")
#     .where(col("current_mrr").isNotNull())
#     .toPandas()["current_mrr"]
# )

# plt.figure(figsize=(8, 5))
# sns.histplot(values, kde=True, bins=40)

# plt.xlabel("current_mrr")
# plt.ylabel("Count")
# plt.title("Histogram + KDE")
# plt.show()


#### Comments
Look at this, we can assumet hat the current_mrr which are kind of outliers maybe business? It is a question, can ask later.

## Create the churn column

Rule:
churn = "Yes": if status is cancelled or expired
or end_date has already passed

churn = "No": if status is active or trial
and the subscription has not ended


In [14]:
df8 = df7.withColumn(
    "churn",
    F.when(
        (F.lower(F.col("status")).isin("cancelled", "canceled", "expired")) |
        (
            F.col("end_date").isNotNull() &
            (F.to_date(F.col("end_date")) <= F.current_date())
        ),
        F.lit("Yes")
    ).otherwise(F.lit("No"))
)

In [15]:
df_silver_subscriptions = df8

## Get all quarantine tables

In [16]:
quarantine_dfs = [
    df1_quarantine,
    df2_quarantine,
    df4_quarantine,
    df5_quarantine,
    df6_quarantine
]

df_quarantine_all = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    quarantine_dfs
)
df_quarantine_all.count()

750000

## Upsert strategy

In [17]:
silver_path = "./silver/subscriptions"

In [18]:
# The columns that we need
silver_cols = df_silver_subscriptions.columns

# Select only silver columns
df_upsert = df_silver_subscriptions.select(*silver_cols)

# Keep only the latest row per subscription_id in the incoming batch
w = Window.partitionBy("subscription_id").orderBy(F.col("created_at").desc(), F.col("ingest_time").desc())

df_upsert = (
    df_upsert
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# If Delta table already exists -> MERGE (upsert)
if DeltaTable.isDeltaTable(spark, silver_path):
    print("Delta table already exists")
    delta_target = DeltaTable.forPath(spark, silver_path)

    (
        delta_target.alias("target")
        .merge(
            df_upsert.alias("source"),
            "target.subscription_id = source.subscription_id"
        )
        # Update only if incoming row is newer
        .whenMatchedUpdate(
            condition="source.ingest_time > target.ingest_time",
            set={c: f"source.{c}" for c in silver_cols}
        )
        .whenNotMatchedInsert(
            values={c: f"source.{c}" for c in silver_cols}
        )
        .execute()
    )

# If Delta table does not exist yet -> initial save
else:
    print("Delta table does not exist yet")
    (
        df_upsert
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )

Delta table does not exist yet


In [23]:
df_subscriptions_silver_read = (
    spark.read
    .format("delta")
    .load(silver_path)
)

df_subscriptions_silver_read.show(5)

ConnectionRefusedError: [Errno 111] Connection refused

In [24]:
spark.stop()

ConnectionRefusedError: [Errno 111] Connection refused